In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from pathlib import Path
from dask.distributed import Client
import sys, os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

sys.path.append("/home/548/cd3022/repos/nowcast-evaluation/src")
import nowcast_preprocess

In [2]:
client = Client(
    n_workers=24,
    threads_per_worker=1
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 24
Total threads: 24,Total memory: 188.56 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39331,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37705,Total threads: 1
Dashboard: /proxy/40873/status,Memory: 7.86 GiB
Nanny: tcp://127.0.0.1:45765,


In [3]:
YEAR = "2025"
MONTH = "01"
DAY = "01"

In [4]:
def preprocess_gso_of(ds):
    ds = nowcast_preprocess.datetime_to_lead_time(ds)
    return ds.sel(
        latitude=slice(-28.5, -35),
        longitude=slice(145, 151.5)
    )[["surface_global_irradiance"]]

gso_of_path = Path(f"/g/data/er8/nowcasts/nwc_20240701_20251231/{YEAR}/{MONTH}/{DAY}")
gso_of_files = [f for f in gso_of_path.glob("*.nc")]
gso_of = xr.open_mfdataset(
    gso_of_files,
    preprocess = preprocess_gso_of,
    concat_dim='time',
    combine='nested',
    data_vars='minimal',
    coords='minimal',
    compat='override',
    parallel=True,
    chunks='auto'
)


def preprocess_truth(ds):
    return ds.sel(
        latitude=slice(-35, -28.5),
        longitude=slice(145, 151.5)
    )[["surface_global_irradiance"]]


truth_path = Path(f"/g/data/rv74/satellite-products/arc/der/himawari-ahi/solar/p1s/latest/{YEAR}/{MONTH}/{DAY}")
truth_files = [f for f in truth_path.glob("*.nc")]
truth = xr.open_mfdataset(
    truth_files,
    preprocess = preprocess_truth,
    concat_dim='time',
    combine='nested',
    data_vars='minimal',
    coords='minimal',
    compat='override',
    parallel=True,
    chunks='auto'
)

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/pyth

In [5]:
def calc_error_metrics(
    fcst: xr.DataArray,
    obs: xr.DataArray,
) -> xr.DataArray:
    """Return RMSE with a single dimension: lead time."""
    # forecast_time must be a lead time (timedelta), not a valid time
    if not np.issubdtype(fcst["forecast_time"].dtype, np.timedelta64):
        raise TypeError(
            f"'{lead_dim}' should hold lead times (timedelta64). If it holds valid "
            f"times, convert it first: lead = valid_time - init_time."
        )

    # Put obs on exactly the forecast grid (guards against float coordinate mismatch)
    obs = obs.sel({"latitude": fcst["latitude"], "longitude": fcst["longitude"]}, method="nearest")
    obs = obs.assign_coords({"latitude": fcst["latitude"], "longitude": fcst["longitude"]})

    rmse_list = []
    mae_list = []
    for lead in fcst["forecast_time"].values:
        f = fcst.sel({"forecast_time": lead})
        valid_times = f["time"].values + lead
        # Observations at each forecast's valid time (NaN where obs are missing)
        o = obs.reindex({"time": valid_times})
        o = o.assign_coords({"time": f["time"].values})

        sq_err = (f - o) ** 2
        mse = sq_err.mean(dim=["latitude", "longitude"], skipna=True)
        rmse = np.sqrt(mse)
        rmse_list.append(rmse)

        mae = f - o
        mae_list.append(mae)

    rmse_array = xr.concat(rmse_list, dim=fcst["forecast_time"]).rename("rmse")
    mae_array = xr.concat(mae_list, dim=fcst["forecast_time"]).rename("mae")

    return xr.merge([rmse_array, mae_array])

In [6]:
metrics = calc_error_metrics(gso_of["surface_global_irradiance"], truth["surface_global_irradiance"])

In [7]:
metrics

<xarray.Dataset> Size: 2GB
Dimensions:        (time: 120, forecast_time: 18, longitude: 326, latitude: 326)
Coordinates:
  * time           (time) datetime64[s] 960B 2025-01-01T17:00:00 ... 2025-01-...
  * forecast_time  (forecast_time) timedelta64[ns] 144B 00:10:00 ... 03:00:00
  * longitude      (longitude) float32 1kB 145.0 145.0 145.0 ... 151.5 151.5
  * latitude       (latitude) float32 1kB -28.5 -28.52 -28.54 ... -34.98 -35.0
    crs            int32 4B 0
Data variables:
    rmse           (forecast_time, time) float64 17kB dask.array<chunksize=(1, 1), meta=np.ndarray>
    mae            (forecast_time, time, latitude, longitude) float64 2GB dask.array<chunksize=(1, 1, 227, 326), meta=np.ndarray>